In [3]:
# Colab only. Locally, pyproject.toml / requirements-lock.txt provide CoolProp.
import sys
if "google.colab" in sys.modules:
    get_ipython().run_line_magic("pip", "install CoolProp")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 46.0 MB/s eta 0:00:00


## Power Output Calculation for Geothermal Well Classes repurposed from oil and gas wells

The power output for each geothermal well class is calculated using the lower bound of the temperature range for that class, with **Class 5** set to 75°C. The bottom-hole temperature is converted to Kelvin, and the specific enthalpy of the saturated liquid at this temperature is determined using CoolProp. The power output is then calculated as \cite{Wight2015}\cite{Santos2022}:

$$
P_{\text{output}} = \dot{m} \times (h_{\text{inlet}} - h_{\text{outlet}}) \times \eta
$$

where $ \dot{m} $ is the mass flow rate, $ h_{\text{inlet}} $ and $ h_{\text{outlet}} $ are the specific enthalpy values at the inlet and outlet, and $ \eta $ is the plant efficiency. Annual energy output is obtained by multiplying the power output by the uptime and total hours in a year. This process is repeated for each class to determine the power and energy output for wells in their respective temperature ranges.

---

### Table: Parameters Used in the repurposed oil-gas wells repurposed geothermal plants

| **Parameter**                 | **Value**               |
|-------------------------------|-------------------------|
| Well Depth $(H)$ \cite{usgs2011}           | 3000 m                 |
| Mass Flow Rate $(\dot{m})$ \cite{Wight2015}\cite{Santos2022}  | 12 kg/s               |
| Plant Efficiency $(\eta)$ \cite{Santos2022}  | 0.09 (9)             |



## References
@techreport{usgs2011,
  author       = {U.S. Geological Survey},
  title        = {Open-File Report 2011-1266},
  year         = {2011},
  institution  = {U.S. Geological Survey},
  url          = {https://pubs.usgs.gov/of/2011/1266/},
  note         = {Accessed: 2025-01-24}
}

@article{wight2015,
  author       = {Wight, N. and Bennett, N.},
  title        = {Geothermal energy from abandoned oil and gas wells using water in combination with a closed wellbore},
  journal      = {Applied Thermal Engineering},
  volume       = {89},
  pages        = {908--915},
  year         = {2015},
  month        = {October},
  day          = {5},
  url          = {https://www.sciencedirect.com/science/article/pii/S1359431115005840},
  note         = {Accessed: 2025-01-24}
}

@article{Santos2022,
   
   author = {L. Santos and A. Dahi Taleghani and D. Elsworth},
   doi = {10.1016/J.RENENE.2022.05.138},
   issn = {0960-1481},
   journal = {Renewable Energy},
   keywords = {Abandoned wells,Geothermal energy,Green energy,Heat extraction,Well integrity},
   month = {7},
   pages = {1288-1302},
   publisher = {Pergamon},
   title = {Repurposing abandoned wells for geothermal energy: Current status and future prospects},
   volume = {194},
   year = {2022},
}


In [4]:
from CoolProp.CoolProp import PropsSI
import math
import pandas as pd


# Define temperature ranges for each class
temperature_classes = {
    "Class 1": 150,  # Lower bound for Class 1
    "Class 2": 130,  # Lower bound for Class 2
    "Class 3": 110,  # Lower bound for Class 3
    "Class 4": 90,   # Lower bound for Class 4
    "Class 5": 73,   # Explicit value for Class 5
}

# Function to calculate power output for a given temperature class
def calculate_geothermal_output(class_name):
    # Input Parameters
    well_depth_meters = 3000  # Depth of the well in meters
    bottom_temperature_celsius = temperature_classes[class_name]  # Lower bound of the class
    fluid_mass_flow_rate_kg_per_s = 12  # Mass flow rate of the fluid in kg/s
    plant_efficiency = 0.09  # Efficiency of the power plant (fractional)
    annual_uptime_fraction = 0.95  # Fraction of time the plant operates per year

    # Convert bottom temperature to Kelvin
    bottom_temperature_kelvin = bottom_temperature_celsius + 273.15  # Convert °C to K
    pressure_pa1=16 * 1e5
    pressure_pa2=4.39 * 1e5
    # Calculate specific enthalpy for saturated liquid water at bottom temperature
    specific_enthalpy_inlet_kj_per_kg = PropsSI('H', 'T', bottom_temperature_kelvin, 'P', pressure_pa1, 'R134a') / 1000  # Convert J/kg to kJ/kg

    specific_enthalpy_outlet_kj_per_kg = PropsSI('H', 'T', 283.15,'P', pressure_pa2, 'R134a') / 1000  # Convert J/kg to kJ/kg
    # Calculate power output (in kW)
    power_output_kW = fluid_mass_flow_rate_kg_per_s * (specific_enthalpy_inlet_kj_per_kg - specific_enthalpy_outlet_kj_per_kg) * plant_efficiency

    # Calculate annual energy output (in kWh)
    hours_in_a_year = 8760
    annual_energy_output_kWh = power_output_kW * annual_uptime_fraction * hours_in_a_year

    # Compile results
    results = {
        "Class": class_name,
        # "Well Depth (m)": well_depth_meters,
        "Fluid Mass Flow Rate (kg/s)": fluid_mass_flow_rate_kg_per_s,
        "Bottom Temperature (°C)": bottom_temperature_celsius,
        "Specific Enthalpy Inlet (kJ/kg)": specific_enthalpy_inlet_kj_per_kg,
        "Specific Enthalpy Outlet (kJ/kg)": specific_enthalpy_outlet_kj_per_kg,
        "Power Output per Well (kW)": power_output_kW,
        "Annual Energy Output (kWh)": annual_energy_output_kWh,
    }

    return results

# Example: Calculate for all classes
all_class_results = [calculate_geothermal_output(class_name) for class_name in temperature_classes]

# Display results for each class
#all_class_results
geothermal_well_df = pd.DataFrame(all_class_results)

# Convert results to a DataFrame
geothermal_well_df_filtered = geothermal_well_df[["Class", "Power Output per Well (kW)"]]

geothermal_well_df_filtered


,Class,Power Output per Well (kW)
0,Class 1,343.448514
1,Class 2,319.343052
2,Class 3,295.353725
3,Class 4,271.167301
4,Class 5,249.928833


# Geothermal Capacity  Estimation for potentioal areas

To estimates the power density and energy production capacity of geothermal reservoirs based on temperature ranges. It involves categorizing geothermal resources into previously defined classes defined by temperature ranges. Middle point of each class is considered as . These representative temperatures reflect the average or midpoint temperatures within the specified range.


$$
\text{Power Density (MW/km²)} = 0.408 \times e^{0.014 \times T}
$$


The plant capacity for each class is then determined by multiplying the calculated power density by the reservoir area, which is assumed to be 10 km² in this scenario.


@misc{nrel2024,
  author       = {National Renewable Energy Laboratory (NREL)},
  title        = {Annual Technology Baseline (ATB) 2024: Geothermal},
  year         = {2024},
  url          = {https://atb.nrel.gov/electricity/2024/geothermal},
  note         = {Accessed: 2025-01-24}
}

In [38]:


# Input data for temperature ranges
temperature_data = {
    "Class": ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5"],
    "Temperature Range (°C)": [">150°C", "130°C - 150°C", "110°C - 130°C", "90°C - 110°C", "<90°C"],
    "Representative Temperature (°C)": [170, 140, 120, 100, 80]  # Estimated midpoints or average temperatures
}

# Convert the data to a DataFrame
Geo_conventional_df = pd.DataFrame(temperature_data)

# Given reservoir area
reservoir_area_km2 = 10  # in km²

# Power density function: Resource potential [MW/km²] = 0.408 * exp(0.014 * Temperature [°C])
Geo_conventional_df["Power Density (MW/km²)"] = Geo_conventional_df["Representative Temperature (°C)"].apply(
    lambda temp: 0.408 * math.exp(0.014 * temp)
)

# Calculate plant capacity for each class
Geo_conventional_df["Average Plant Capacity (MW)"] = Geo_conventional_df["Power Density (MW/km²)"] * reservoir_area_km2


sq_miles_per_sq_km = 1 / 2.58999

# Example power density in MW/km²
power_density_mw_per_km2 = 3.832479  # Example value

# Convert to MW/sq.mile
power_density_mw_per_sq_mile = power_density_mw_per_km2 / sq_miles_per_sq_km

total_capacity=power_density_mw_per_sq_mile*32050/1000
print(total_capacity,power_density_mw_per_sq_mile)

Geo_conventional_df




318.1309372409805 9.926082285209999


,Class,Temperature Range (°C),Representative Temperature (°C),Power Density (MW/km²),Average Plant Capacity (MW)
0,Class 1,>150°C,170,4.408400,44.084004
1,Class 2,130°C - 150°C,140,2.896525,28.965254
2,Class 3,110°C - 130°C,120,2.189147,21.891468
3,Class 4,90°C - 110°C,100,1.654522,16.545216
4,Class 5,<90°C,80,1.250461,12.504605


## Assumptions

### Size of a Geothermal Development :
Each geothermal facility requires **10 km²** for the plant infrastructure.

### Buffer Distance Between Developments:
Each geothermal facility requires a **500 m buffer radius** for optimal spacing.

### Calculation of Buffer Area:
Buffer area is calculated using the formula for the area of a circle:

$$
\text{Buffer area} = \pi \times (\text{radius})^2 = \pi \times (500 \text{ m})^2
$$

### Total Area Required:
The total area required for a typical geothermal power plant includes the infrastructure area and the buffer area:

$$
\text{Total Area Required} = \text{Plant Infrastructure Area} + \text{Buffer Area}
$$

In [94]:


# Input data
# GIS_data = {
#     "Class": ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5 and 999"],
#     "Tribal Lands (sq. miles)": [800, 3730, 7413, 6364, 6851],
#     "DACs (sq. miles)": [360, 4050, 5225, 4880, 6543],
#     "Total Area (sq. miles)": [2150, 25661, 25642, 24670, 36359],
#     "Oil Wells Count": [20, 2200, 12000, 2000, 2300]
# }
# Input data
# GIS_data = {
#     "Class": ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5 and 999"],
#     "Tribal Lands (sq. miles)": [1300, 4173, 7413, 6364, 6851],
#     "DACs (sq. miles)": [550, 4050, 5225, 4880, 6543],
#     "Total Area (sq. miles)": [32050, 144141, 196118, 221513, 236160],
#     "Oil Wells Count": [20, 2200, 12000, 2000, 2300]
# }

import pandas as pd

# Creating the DataFrame
GIS_data = {
    "Class": ["Class 1", "Class 2", "Class 3", "Class 4", "Class 5 and 999"],
    "Total Potential (sq. miles)": [32050, 144141, 196118, 221513, 236160],
    "Hub Zones (sq. miles)": [10090, 38776, 74051, 22641, 31320],
    "Tribal Lands (sq. miles)": [1838, 10832, 48026, 18729, 18845],
    "Oil Wells Count": [20, 2200, 12000, 2000, 2300],
    "Oil Wells in Hub Zones (#)": [1, 201, 5964, 334, 663],
    "Oil Wells in Tribal Communities (#)": [0, 22, 5752, 290, 370]
}



# Assumptions
plant_infrastructure_area_km2 = 10  # Plant infrastructure area in km²
buffer_area_meter = 500  # Buffer radius in meters

# Convert areas to square miles
plant_infrastructure_area_sq_miles = plant_infrastructure_area_km2 / 2.59  # Convert 10 km² to square miles
buffer_radius_miles = buffer_area_meter / 1609.34  # Convert 500 meters to miles

# Calculate buffer area in square miles using the formula: π × (radius)^2
buffer_area_sq_miles = math.pi * (buffer_radius_miles**2)

total_area_per_facility_sq_miles = plant_infrastructure_area_sq_miles + buffer_area_sq_miles

# Convert data to DataFrame
df = pd.DataFrame(GIS_data)

# Calculate the maximum number of geothermal facilities for each land category
for land_type in ["Tribal Lands", "Hub Zones", "Total Potential"]:
    df[f"{land_type} Max Facilities"] = (df[f"{land_type} (sq. miles)"] // total_area_per_facility_sq_miles).astype(int)

# Extract necessary columns for max facilities
df["Class"] = df["Class"].replace({"Class 5 and 999": "Class 5"})
max_facilities_df = df[[
    "Class",
    "Tribal Lands Max Facilities",
    "Hub Zones Max Facilities",
    "Total Potential Max Facilities",
    "Oil Wells Count",
    "Oil Wells in Hub Zones (#)",
    "Oil Wells in Tribal Communities (#)"
]]

# Merge plant capacity with max facilities
Geo_conventional_df_ = Geo_conventional_df.merge(max_facilities_df, on="Class")

# Calculate total capacity for each land category
for land_type in ["Tribal Lands", "Hub Zones", "Total Potential"]:
    Geo_conventional_df_[f"{land_type} (GW)"] = (
        Geo_conventional_df_["Average Plant Capacity (MW)"] * Geo_conventional_df_[f"{land_type} Max Facilities"]/1000
    )



# Filter the DataFrame to keep only the desired columns
columns_to_keep = [
    "Class",
    "Temperature Range (°C)",
    "Tribal Lands Max Facilities",
    "Hub Zones Max Facilities",
    "Total Potential Max Facilities",
    "Tribal Lands (GW)",
    "Hub Zones (GW)",
    "Total Potential (GW)",
    "Oil Wells Count",
    "Oil Wells in Hub Zones (#)",
    "Oil Wells in Tribal Communities (#)"

]

filtered_geothermal_data = Geo_conventional_df_[columns_to_keep]

# Merge filtered geothermal data with power output data
power_output_df = pd.DataFrame(geothermal_well_df_filtered)
final_df = filtered_geothermal_data.merge(power_output_df, on="Class", how="left")

# Add a column for the total power output based on oil wells
final_df["Oil Wells (GW)"] = (
    final_df["Oil Wells Count"] * final_df["Power Output per Well (kW)"] / 1000000
)
final_df["Oil Wells (GW) in Hub Zones"] = (
    final_df["Oil Wells in Hub Zones (#)"] * final_df["Power Output per Well (kW)"] / 1000000
)
final_df["Oil Wells (GW) on Tribal Lands"] = (
    final_df[ "Oil Wells in Tribal Communities (#)"] * final_df["Power Output per Well (kW)"] / 1000000
)


# Drop unnecessary columns
final_df = final_df.drop(columns=[ "Power Output per Well (kW)","Oil Wells in Hub Zones (#)", "Oil Wells in Tribal Communities (#)"])
final_df_ = final_df.drop([3, 4], axis=0)
# Display the final DataFrame
print(final_df['Total Potential (GW)'].sum())
print(final_df_['Total Potential (GW)'].sum())
final_df_

3962.0618764388673
2372.823546316849


,Class,Temperature Range (°C),Tribal Lands Max Facilities,Hub Zones Max Facilities,Total Potential Max Facilities,Tribal Lands (GW),Hub Zones (GW),Total Potential (GW),Oil Wells Count,Oil Wells (GW),Oil Wells (GW) in Hub Zones,Oil Wells (GW) on Tribal Lands
0,Class 1,>150°C,441,2423,7696,19.441046,106.815541,339.270492,20,0.006869,0.000343,0.000000
1,Class 2,130°C - 150°C,2601,9311,34613,75.338627,269.695484,1002.574351,2200,0.702555,0.064188,0.007026
2,Class 3,110°C - 130°C,11532,17782,47095,252.452413,389.274090,1030.978703,12000,3.544245,1.761490,1.698875


In [95]:
# Check actual column names for debugging
print("Columns in final_df_:", final_df_.columns.tolist())

# Compute the total row
total_row = final_df_.sum(numeric_only=True)
total_row["Class"] = "Total"  # Label for total row

# Append the total row using pd.concat()
final_df_sumadded = pd.concat([final_df_, pd.DataFrame([total_row])], ignore_index=True)



# Select only relevant columns for the LaTeX table
columns_for_table = [
    "Class",
    "Total Potential (GW)",
    "Hub Zones (GW)",
    "Tribal Lands (GW)",
    "Oil Wells (GW)",
    "Oil Wells (GW) in Hub Zones",
    "Oil Wells (GW) on Tribal Lands"
]

# Create a copy of the DataFrame to avoid SettingWithCopyWarning
final_df_for_table = final_df_sumadded[columns_for_table].copy()

# Round numerical values and convert to string to remove .0 decimals
for col in final_df_for_table.columns[1:]:
    final_df_for_table.loc[:, col] = final_df_for_table[col].round(1).astype(str)

# Convert DataFrame to LaTeX with multi-line headers (escaping curly braces properly)
latex_table = final_df_for_table.to_latex(
    index=False,
    column_format="|c|c|c|c|c|c|c|",  # Ensure proper alignment
    header=[
        "Class",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Total \\\\ Potential (GW)\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Hub \\\\ Zones (GW)\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Tribal \\\\ Lands (GW)\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Oil \\\\ Wells (GW)\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Oil Wells (GW) in \\\\ Hub Zones\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Oil Wells (GW) \\\\ on Tribal Lands\\end{{tabular}}}}"
    ],
    escape=False  # Allows LaTeX syntax inside the table
)

# Format the LaTeX output properly
formatted_latex_table = f"""
\\begin{{table}}[H]
\\caption{{Technical Geothermal Potential Capacity of Selected Area and Investigated Categories}}
\\label{{tab:result1}}
\\resizebox{{\\textwidth}}{{!}}{{%
{latex_table}
}}
\\end{{table}}
"""

# Print the properly formatted LaTeX table
print(formatted_latex_table)



Columns in final_df_: ['Class', 'Temperature Range (°C)', 'Tribal Lands Max Facilities', 'Hub Zones Max Facilities', 'Total Potential Max Facilities', 'Tribal Lands (GW)', 'Hub Zones (GW)', 'Total Potential (GW)', 'Oil Wells Count', 'Oil Wells (GW)', 'Oil Wells (GW) in Hub Zones', 'Oil Wells (GW) on Tribal Lands']

\begin{table}[H]
\caption{Technical Geothermal Potential Capacity of Selected Area and Investigated Categories}
\label{tab:result1}
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|}
\toprule
Class & \textbf{\begin{tabular}[c]{@{}c@{}}Total \\ Potential (GW)\end{tabular}} & \textbf{\begin{tabular}[c]{@{}c@{}}Hub \\ Zones (GW)\end{tabular}} & \textbf{\begin{tabular}[c]{@{}c@{}}Tribal \\ Lands (GW)\end{tabular}} & \textbf{\begin{tabular}[c]{@{}c@{}}Oil \\ Wells (GW)\end{tabular}} & \textbf{\begin{tabular}[c]{@{}c@{}}Oil Wells (GW) in \\ Hub Zones\end{tabular}} & \textbf{\begin{tabular}[c]{@{}c@{}}Oil Wells (GW) \\ on Tribal Lands\end{tabular}} \\
\midrule
Class 1 & 3

<ipython-input-95-e0b41e7f8ca0>:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['339.3' '1002.6' '1031.0' '2372.8']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_df_for_table.loc[:, col] = final_df_for_table[col].round(1).astype(str)
<ipython-input-95-e0b41e7f8ca0>:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['106.8' '269.7' '389.3' '765.8']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_df_for_table.loc[:, col] = final_df_for_table[col].round(1).astype(str)
<ipython-input-95-e0b41e7f8ca0>:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['19.4' '75.3' '252.5' '347.2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  fin

In [93]:


# Select relevant columns for the LaTeX table
columns_for_table = [
    "Class",
    "Total Potential Max Facilities",
    "Hub Zones Max Facilities",
    "Tribal Lands Max Facilities"
]

# Create a copy of the DataFrame to avoid SettingWithCopyWarning
final_df_for_table = final_df_[columns_for_table].copy()

# Merge with Geo_conventional_df
final_df_for_table = final_df_for_table.merge(
    Geo_conventional_df[["Class", "Average Plant Capacity (MW)"]],  # Ensure "Class" is included
    on="Class",
    how="left"
)



# Convert "Max Facilities" columns to integer values
for col in ["Total Potential Max Facilities", "Hub Zones Max Facilities", "Tribal Lands Max Facilities"]:
    final_df_for_table[col] = final_df_for_table[col].astype(int)

# Compute the total row
total_row = final_df_for_table.sum(numeric_only=True)
total_row["Class"] = "Total"  # Label for total row

# Append the total row using pd.concat()
final_df_for_table = pd.concat([final_df_for_table, pd.DataFrame([total_row])], ignore_index=True)

print(final_df_for_table)
# Round numbers for clean formatting
for col in final_df_for_table.columns[1:]:  # Skip "Class" column
    if col in ["Total Potential Max Facilities", "Hub Zones Max Facilities", "Tribal Lands Max Facilities"]:
        final_df_for_table[col] = final_df_for_table[col].astype(int)  # Keep integers
    else:
        final_df_for_table[col] = final_df_for_table[col].round(2).astype(str)  # Round floats to 1 decimal place


# Convert DataFrame to LaTeX with properly escaped multi-line headers
latex_table = final_df_for_table.to_latex(
    index=False,
    column_format="|c|c|c|c|c|",  # Ensure proper alignment
    header=[
        "Class",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Max Potential\\\\ Facilities\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Hub Zones \\\\ Max Facilities\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Tribal Lands \\\\ Max Facilities\\end{{tabular}}}}",
        "\\textbf{{\\begin{{tabular}}[c]{{@{{}}c@{{}}}}Average Plant\\\\ Capacity (MW)\\end{{tabular}}}}"
    ],
    escape=False  # Allows LaTeX syntax inside the table
)

# Format the LaTeX output properly
formatted_latex_table = f"""
\\begin{{table}}[H]
\\centering
\\caption{{Maximum Number of Potential Geothermal Facilities in the Selected Area and by Investigated Categories}}
\\label{{tab:result2}}
\\resizebox{{\\textwidth}}{{!}}{{%
{latex_table}
}}
\\end{{table}}
"""

# Print the properly formatted LaTeX table
print(formatted_latex_table)




     Class  Total Potential Max Facilities  Hub Zones Max Facilities  \
0  Class 1                          7696.0                    2423.0   
1  Class 2                         34613.0                    9311.0   
2  Class 3                         47095.0                   17782.0   
3    Total                         89404.0                   29516.0   

   Tribal Lands Max Facilities  Average Plant Capacity (MW)  
0                        441.0                    44.084004  
1                       2601.0                    28.965254  
2                      11532.0                    21.891468  
3                      14574.0                    94.940726  

\begin{table}[H]
\centering
\caption{Maximum Number of Potential Geothermal Facilities in the Selected Area and by Investigated Categories}
\label{tab:result2}
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|}
\toprule
Class & \textbf{\begin{tabular}[c]{@{}c@{}}Max Potential\\ Facilities\end{tabular}} & \textbf{\begin{t

In [ ]:
# Constants for calculations
geothermal_gradient = 0.0311  # °C/m
surface_temperature = 23.005  # °C
depths = [4200, 4900, 6000, 6800]  # Well depths in meters
mass_flow_rates = [2.5, 5, 10, 15]  # Mass flow rates in kg/s
specific_enthalpy_inlet = 544.5  # kJ/kg (assumed for 130°C)
specific_enthalpy_outlet = 209.3  # kJ/kg (assumed for 50°C)
plant_efficiency = 0.12  # Binary plant efficiency


# Results storage
results = []

# Calculations
for depth, mass_flow_rate in zip(depths, mass_flow_rates):
    # Calculate fluid temperature at the bottom
    bottom_temperature = surface_temperature + geothermal_gradient * depth



    # Net power output
    power_output = mass_flow_rate * (specific_enthalpy_inlet - specific_enthalpy_outlet) * plant_efficiency


    results.append({
        "Depth (m)": depth,
        "Mass Flow Rate (kg/s)": mass_flow_rate,
        "Bottom Temp (°C)": bottom_temperature,


        "Power Output (kW)": power_output,

    })

import pandas as pd
results_df = pd.DataFrame(results)



results_df

,Depth (m),Mass Flow Rate (kg/s),Bottom Temp (°C),Power Output (kW)
0,4200,2.5,153.625,100.56
1,4900,5.0,175.395,201.12
2,6000,10.0,209.605,402.24
3,6800,15.0,234.485,603.36
